# Stage 0 · CPU notebook

Source readiness, verified spectral cache and explicit projection reporting. No model inference or GPU runtime is required. The GPU notebook shares the saved plans.


## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime is sufficient for all current cells.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

RELEASE_COMMIT = '6664bc55df4b8e16136fac29749b609b6cd0d352'
REPOSITORY_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
# Only the dedicated Colab checkout is pinned; local work is never switched.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').is_dir():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
    changed = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True)
    if changed.strip():
        raise RuntimeError('The dedicated checkout has local changes. Preserve them before selecting a release.')
    subprocess.run(['git', 'fetch', 'origin', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Select the repository checkout before running the workflow.')

def run_stage0(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    return subprocess.run(command, cwd=REPO_DIR, check=True)


## 2. Select and prepare
Bind the SFT/GRPO prepared sources. Optional inputs remain distinguishable from verified endpoint adapters.

In [ ]:
# Select existing prepared products; paths are relative to ARTIFACT_ROOT.
SELECTION = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'source-selection.json'
settings = {
    'sources': {
        'sft': 'plans/pilot3-inputs-v1/activation.prepared.json',
        'grpo': 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
        'random': 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
    },
    # Optional product paths must point to explicit file-hash completion receipts.
    'products': {},
}
SELECTION.parent.mkdir(parents=True, exist_ok=True)
if SELECTION.exists() and json.loads(SELECTION.read_text()) != settings:
    raise RuntimeError('Saved source selection differs. Inspect it before selecting another plan name.')
SELECTION.write_text(json.dumps(settings, indent=2) + '\n')
run_stage0('prepare', '--selection', SELECTION, '--name', STAGE0_NAME)


## 3. Audit or resume
Verify selected config, seal and adapter payloads; compare actual zero-initialization tensors. Sealed source units are reused. No optimizer-state replay, model download or inference.

In [ ]:
run_stage0('audit', '--config', PLAN)


## 4. Restore saved status
Safe in a fresh kernel after section 1. Reports audit evidence, missing prerequisites and the cumulative 20-unit budget. Unknown cost is not zero cost.

In [ ]:
run_stage0('status', '--config', PLAN)


## 5. Prepare the local spectral cache
Input: completed source audit and pinned local safetensor files. Output: an immutable cache plan. This verifies the same FP32 base hash recorded by training; it neither downloads nor constructs a model. Stage weights using the guide before this cell.

In [ ]:
AUDIT_RESULT = ARTIFACT_ROOT / 'runs' / STAGE0_NAME / 'results' / 'source-audit.json'
audit = json.loads(AUDIT_RESULT.read_text())
revision = audit['sources']['sft']['identity']['model_revision']
WEIGHTS_DIR = ARTIFACT_ROOT / 'inputs' / 'models' / revision
SPECTRAL_NAME = STAGE0_NAME + '-spectral-v1'
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / SPECTRAL_NAME / 'stage0.spectral.json'
if not (WEIGHTS_DIR / 'config.json').is_file():
    raise RuntimeError('Stage the pinned local weight snapshot at ' + str(WEIGHTS_DIR) + '; see docs/stage0-geometry.md.')
run_stage0('spectral-prepare', '--audit', PLAN, '--weights-dir', WEIGHTS_DIR, '--name', SPECTRAL_NAME)


## 6. Measure one cache unit
Output: one sealed module decomposition, measured seconds and retained bytes. This is a bounded first measurement, not a guaranteed full-run ETA. Rerunning reuses the sealed unit.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN, '--max-new-units', 1)


## 7. Complete or resume the cache
Decompose only missing modules. Each sealed unit retains all singular values and enough bases for k30/k50/k70 and the top-m intruder rule. Source identities, reconstruction and orthogonality are checked. No trained overlap numbers are revealed.

In [ ]:
run_stage0('spectral-run', '--config', SPECTRAL_PLAN)


## 8. Verify saved cache
Safe after restart once section 1 is run and SPECTRAL_PLAN is restored. Output: verified cache summary; later edit stages can consume these bases without opening the overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
run_stage0('spectral-verify', '--config', SPECTRAL_PLAN)


## 9. Optional trained-overlap reveal
Leave disabled if preserving the conditional gradient-to-overlap prediction. To reveal, supply an explicit review recording either frozen-before-reveal predictions or exploratory status; the guide documents its fields. Outputs: per-module JSONL/CSV, plot and completion receipt.

In [ ]:
REVEAL_TRAINED_OVERLAPS = False
OVERLAP_REVIEW = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'overlap-review.json'
if REVEAL_TRAINED_OVERLAPS:
    run_stage0('spectral-report', '--config', SPECTRAL_PLAN, '--review', OVERLAP_REVIEW,
               '--name', STAGE0_NAME + '-principal-overlap-v1')
else:
    print('Trained overlaps remain unrevealed. Cache is ready for edit construction.')


## 10. Prepare the principal intervention catalogue
Input: verified spectral cache and audited endpoint adapters. Output: immutable edit plan. This uses the cache without opening the trained-overlap report.

In [ ]:
SPECTRAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-spectral-v1') / 'stage0.spectral.json'
EDIT_NAME = STAGE0_NAME + '-principal-edits-v1'
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / EDIT_NAME / 'stage0.edits.json'
run_stage0('edits-prepare', '--spectral', SPECTRAL_PLAN, '--name', EDIT_NAME)


## 11. Build or resume adapters
Output: ten separate PEFT adapters, per-module check records and saved random bases. Original source adapters are never modified. Completed module units are reused after interruption. CPU only; no model is loaded.

In [ ]:
run_stage0('edits-run', '--config', EDIT_PLAN)


## 12. Verify the catalogue
Safe after restart following section 1. Output: paths, operations and verified payload identities. Construction completion does not mean GPU qualification or evaluation has passed.

In [ ]:
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-principal-edits-v1') / 'stage0.edits.json'
run_stage0('edits-verify', '--config', EDIT_PLAN)


## Next stage
Ticket 04 adds bounded GPU qualification and measured-budget approval. Keep the GPU disconnected until that stage is available. Conditional intruder edits V6/V6c are a separate later extension.